In [16]:
import os
import time
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =============================================================================
# 1. CONFIGURATION & OFFLINE PATHS
# =============================================================================
COMPETITION_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

# Private dataset path containing your 4000+ LLM soft target probabilities
LLM_DATASET_CSV = Path("/kaggle/input/datasets/tnshpp/llm-targets/llm_labels_gemma-26b.csv")

OUTPUT_DIR = Path("./dinov2_pipeline_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pipeline Improvements Hyperparameters
N_FOLDS = 3
SEED = 2026
UNFREEZE_LAST = 6               # Fine-tune top 6 transformer blocks[cite: 1]
EPOCHS = 10
BATCH_SIZE = 8
LR_HEAD = 1e-3
LR_BACKBONE = 5e-6              # Reduced backbone LR to preserve DINO features[cite: 3, 5]
WEIGHT_DECAY = 0.02
CROP_MM = 140.0                 # Standard physical crop normalization (mm)[cite: 1]
IMG_SIZE = 378                  # Higher spatial resolution[cite: 1]
K_WINDOWS = 16                  # Window count for dynamic depth attention[cite: 1]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture"
]

# 6 Canonical View Slots
SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
N_SLOT = len(SLOTS)


In [17]:

# =============================================================================
# 2. SLIDING WINDOW (2.5D TRIPLETS) DATASET LOADER
# =============================================================================
def load_and_preprocess_windows(series_dir, crop_mm=CROP_MM, out_size=IMG_SIZE, k_windows=K_WINDOWS):
    """
    Extracts overlapping 3-slice sliding window triplets (z-1, z, z+1) across the volume depth
    to prevent focal pathology dilution.
    """
    dcm_files = sorted(Path(series_dir).glob("*.dcm"))
    if not dcm_files or len(dcm_files) < 3:
        return None

    # Load all slice arrays and pixel spacings
    slices = []
    px_spacing = 0.5
    for f in dcm_files:
        try:
            ds = pydicom.dcmread(str(f), force=True)
            arr = ds.pixel_array.astype(np.float32)
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            arr = arr * slope + intercept
            
            ps = getattr(ds, 'PixelSpacing', None)
            if ps is not None:
                px_spacing = float(ps[0])
            slices.append(arr)
        except Exception:
            continue

    if len(slices) < 3:
        return None

    vol = np.stack(slices)  # [Z, H, W]
    z_len, h, w = vol.shape

    # Physical Crop Normalization (mm -> px)[cite: 1, 3]
    want = int(round(crop_mm / px_spacing))
    if 16 < want < min(h, w):
        cy, cx = h // 2, w // 2
        half = want // 2
        vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    # Sample window center indices across depth
    centers = np.linspace(1, z_len - 2, k_windows).astype(int)
    
    windows = []
    for c in centers:
        # Construct 2.5D RGB Triplet: (z-1, z, z+1)[cite: 1, 3]
        triplet = vol[c - 1:c + 2]  # [3, H, W]
        t = torch.from_numpy(np.ascontiguousarray(triplet)).unsqueeze(0)
        t = F.interpolate(t, size=(out_size, out_size), mode='bilinear', align_corners=False)
        windows.append(t.squeeze(0))

    out_tensor = torch.stack(windows)  # [K_WINDOWS, 3, H, W]
    return (out_tensor * 255.0).round().clamp(0, 255).to(torch.uint8)


class DINOv2EnhancedDataset(Dataset):
    def __init__(self, df, series_df, data_dir):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.data_dir = Path(data_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_uid = str(row["StudyInstanceUID"])
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == study_uid]

        # Primary slot extraction (Fluid Sagittal / Primary view)
        window_tensor = torch.zeros((K_WINDOWS, 3, IMG_SIZE, IMG_SIZE), dtype=torch.uint8)
        mask = 0.0

        for slot_idx, (name, plane, fluid, fs) in enumerate(SLOTS):
            cand = study_series[(study_series["Anatomical_Plane"] == plane) & 
                                (study_series["Fat_Suppression"] == (1 if fs else 0))]
            if not cand.empty:
                chosen = cand.sort_values("n_slices", ascending=False).iloc[0] if "n_slices" in cand.columns else cand.iloc[0]
                s_uid = str(chosen["SeriesInstanceUID"])
                
                s_path = self.data_dir / study_uid / s_uid
                imgs = load_and_preprocess_windows(s_path)
                if imgs is not None:
                    window_tensor = imgs
                    mask = 1.0
                    break

        labels = torch.tensor(row[TARGETS].values.astype(np.float32))
        return window_tensor, mask, labels, study_uid

# =============================================================================
# 3. DINOv2 WITH DYNAMIC WINDOW SLICE-ATTENTION (RAPTOR-STYLE)
# =============================================================================
class DINOv2WindowAttentionHead(nn.Module):
    """
    Process window triplet features through per-finding dynamic attention pooling[cite: 1].
    Assigns finding-specific attention weights across window centers[cite: 1].
    """
    def __init__(self, embed_dim=384, num_classes=12, hidden_dim=256, dropout=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim * 2)
        
        self.attn = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        self.cls_weight = nn.Parameter(torch.zeros(num_classes, embed_dim * 2))
        self.cls_bias = nn.Parameter(torch.zeros(num_classes))
        nn.init.trunc_normal_(self.cls_weight, std=0.02)

    def forward(self, window_features):
        # window_features: [Batch, K_windows, Feature_Dim]
        h = self.norm(window_features)
        
        # Calculate dynamic window weights per finding: [Batch, K_windows, Num_Classes]
        attn_weights = torch.softmax(self.attn(h), dim=1)
        
        # Pool window features per target finding
        pooled_context = torch.einsum('bkn,bkf->bnf', attn_weights, h)
        logits = (pooled_context * self.cls_weight.unsqueeze(0)).sum(-1) + self.cls_bias
        return logits


class EnhancedDINOv2KneeModel(nn.Module):
    def __init__(self, unfreeze_last=UNFREEZE_LAST, num_classes=12):
        super().__init__()
        # Load dinov2_vits14 natively via PyTorch Hub (bypassing HF import bugs)
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', verbose=False)
        
        for p in self.backbone.parameters():
            p.requires_grad = False
            
        if unfreeze_last > 0:
            for blk in self.backbone.blocks[-unfreeze_last:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in self.backbone.norm.parameters():
                p.requires_grad = True

        embed_dim = self.backbone.embed_dim
        self.head = DINOv2WindowAttentionHead(embed_dim=embed_dim, num_classes=num_classes)
        
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, windows):
        # windows: [Batch, K_windows, 3_channels, H, W]
        B, K, C, H, W = windows.shape
        x = windows.view(B * K, C, H, W).float().div_(255.0)
        x = (x - self.mean) / self.std
        
        if not any(p.requires_grad for p in self.backbone.parameters()):
            with torch.no_grad():
                features = self.backbone.forward_features(x)
        else:
            features = self.backbone.forward_features(x)
            
        cls_token = features["x_norm_clstoken"]
        patch_tokens = features["x_norm_patchtokens"].mean(1)
        combined = torch.cat([cls_token, patch_tokens], dim=1).view(B, K, -1)
        
        return self.head(combined)

# =============================================================================
# 4. POST-PROCESSING: PERCENTILE RANK CALIBRATION
# =============================================================================
def to_percentile_ranks(arr):
    """
    Converts continuous raw probability scores to percentile ranks [0, 1].
    Essential for maximizing macro ROC-AUC[cite: 1, 3].
    """
    if len(arr) <= 1:
        return arr
    return (pd.DataFrame(arr).rank(method="average").to_numpy() - 1.0) / (len(arr) - 1.0)

# =============================================================================
# 5. TRAINING AND EVALUATION LOOPS
# =============================================================================
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0
    for windows, mask, labels, _ in loader:
        windows, labels = windows.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(windows)
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
        
    return total_loss / len(loader.dataset)


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds, targets, uids = [], [], []
    
    for windows, mask, labels, batch_uids in loader:
        windows, labels = windows.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(windows)
            loss = criterion(logits, labels)
            probs = torch.sigmoid(logits)
            
        total_loss += loss.item() * len(labels)
        preds.append(probs.cpu().numpy())
        targets.append(labels.cpu().numpy())
        uids.extend(batch_uids)
        
    y_pred = np.vstack(preds)
    y_true = np.vstack(targets)
    
    # Calculate Macro ROC-AUC using percentile ranks
    y_pred_ranked = to_percentile_ranks(y_pred)
    y_true_binary = (y_true >= 0.5).astype(int)
    
    aucs = []
    for i in range(len(TARGETS)):
        if len(np.unique(y_true_binary[:, i])) > 1:
            aucs.append(roc_auc_score(y_true_binary[:, i], y_pred_ranked[:, i]))
    macro_auc = np.mean(aucs) if aucs else 0.5
    
    return total_loss / len(loader.dataset), macro_auc, y_pred, y_true, uids

# =============================================================================
# 6. PIPELINE EXECUTION
# =============================================================================
def run_pipeline():
    print(f"Initializing Enhanced DINOv2 Pipeline on {DEVICE}")
    
    # 1. Load Radiologist Golden Benchmark Studies (58 studies)[cite: 3]
    golden_df = pd.read_csv(COMPETITION_ROOT / "train.csv").dropna(subset=TARGETS).reset_index(drop=True)
    golden_uids = set(golden_df["StudyInstanceUID"].astype(str))
    print(f"[+] Loaded {len(golden_df)} Radiologist Golden Benchmark studies from COMPETITION_ROOT")

    # 2. Load Private LLM Labels Dataset (4000+ studies)[cite: 3]
    llm_df = pd.read_csv(LLM_DATASET_CSV).dropna(subset=TARGETS).reset_index(drop=True)
    
    # Strictly exclude Golden Benchmark studies from the LLM training dataset[cite: 3]
    llm_df = llm_df[~llm_df["StudyInstanceUID"].astype(str).isin(golden_uids)].reset_index(drop=True)
    print(f"[+] Loaded {len(llm_df)} LLM-labeled training studies (after removing golden overlaps)")

    series_df = pd.read_csv(COMPETITION_ROOT / "train_series.csv")
    series_df["StudyInstanceUID"] = series_df["StudyInstanceUID"].astype(str)
    series_df["SeriesInstanceUID"] = series_df["SeriesInstanceUID"].astype(str)

    golden_ds = DINOv2EnhancedDataset(golden_df, series_df, COMPETITION_ROOT / "train_series")
    golden_loader = DataLoader(golden_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    oof_predictions = np.zeros((len(llm_df), len(TARGETS)), dtype=np.float32)
    oof_targets = llm_df[TARGETS].values.astype(np.float32)
    study_order = np.array(llm_df["StudyInstanceUID"].astype(str).tolist())
    
    golden_preds_folds = []
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    history = {"train_loss": [], "val_loss": [], "val_auc": []}

    for fold, (train_idx, val_idx) in enumerate(kf.split(llm_df)):
        print(f"\n==================== FOLD {fold + 1}/{N_FOLDS} ====================")
        train_sub = llm_df.iloc[train_idx].reset_index(drop=True)
        val_sub = llm_df.iloc[val_idx].reset_index(drop=True)
        
        train_ds = DINOv2EnhancedDataset(train_sub, series_df, COMPETITION_ROOT / "train_series")
        val_ds = DINOv2EnhancedDataset(val_sub, series_df, COMPETITION_ROOT / "train_series")
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
        
        model = EnhancedDINOv2KneeModel(unfreeze_last=UNFREEZE_LAST).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss()
        
        param_groups = [{'params': model.head.parameters(), 'lr': LR_HEAD}]
        if UNFREEZE_LAST > 0:
            backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
            if backbone_params:
                param_groups.append({'params': backbone_params, 'lr': LR_BACKBONE})
                
        optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
        scaler = torch.amp.GradScaler('cuda')
        
        best_auc, best_preds = 0.0, None
        
        for epoch in range(1, EPOCHS + 1):
            t0 = time.time()
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
            val_loss, val_auc, val_preds, _, _ = validate_epoch(model, val_loader, criterion)
            elapsed = time.time() - t0
            
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_auc"].append(val_auc)
            
            print(f"Ep {epoch:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Time: {elapsed:.1f}s")
            
            if val_auc > best_auc:
                best_auc = val_auc
                best_preds = val_preds
                torch.save(model.state_dict(), OUTPUT_DIR / f"dinov2_enhanced_fold{fold}.pt")
                
        oof_predictions[val_idx] = best_preds

        # Evaluate current fold checkpoint on Golden Benchmark set
        model.load_state_dict(torch.load(OUTPUT_DIR / f"dinov2_enhanced_fold{fold}.pt"))
        _, _, g_pred, _, _ = validate_epoch(model, golden_loader, criterion)
        golden_preds_folds.append(g_pred)

    # Save Out-Of-Fold predictions in .npz format[cite: 3]
    npz_path = OUTPUT_DIR / "dinov2_oof_predictions.npz"
    np.savez_compressed(npz_path, values=oof_predictions, study_uids=study_order, targets=oof_targets)
    print(f"\n[+] Saved main out-of-fold predictions to: {npz_path}")

    # Golden Benchmark Ensemble Predictions
    golden_ensemble_preds = np.mean(golden_preds_folds, axis=0)
    golden_targets = golden_df[TARGETS].values.astype(np.float32)
    
    # Save & Display Golden Benchmark Performance Report
    generate_and_save_golden_report(golden_ensemble_preds, golden_targets)
    
    # Save Loss Graph Plot
    plot_loss_curves(history)

# =============================================================================
# 7. REPORTING & VISUALIZATION
# =============================================================================
def generate_and_save_golden_report(preds, targets):
    print("\n" + "=" * 70)
    print(" RADIOLOGIST GOLDEN BENCHMARK PERFORMANCE REPORT (58 STUDIES) ")
    print("=" * 70)
    
    # Apply percentile rank transformation[cite: 1, 3]
    ranked_preds = to_percentile_ranks(preds)
    y_true_binary = (targets >= 0.5).astype(int)
    
    metrics = []
    for i, target in enumerate(TARGETS):
        target_bce = F.binary_cross_entropy_with_logits(
            torch.tensor(preds[:, i]).logit(), 
            torch.tensor(targets[:, i])
        ).item()
        
        if len(np.unique(y_true_binary[:, i])) > 1:
            target_auc = roc_auc_score(y_true_binary[:, i], ranked_preds[:, i])
        else:
            target_auc = 0.5
            
        metrics.append({"Target Finding": target, "Loss (BCE)": target_bce, "ROC-AUC": target_auc})

    report_df = pd.DataFrame(metrics)
    print(report_df.to_string(index=False))
    
    macro_auc = report_df["ROC-AUC"].mean()
    mean_loss = report_df["Loss (BCE)"].mean()
    
    print("-" * 70)
    print(f" GOLDEN BENCHMARK MACRO ROC-AUC : {macro_auc:.4f}")
    print(f" GOLDEN BENCHMARK MEAN BCE LOSS : {mean_loss:.4f}")
    print("=" * 70)

    report_csv_path = OUTPUT_DIR / "golden_benchmark_metrics.csv"
    report_df.to_csv(report_csv_path, index=False)
    print(f"[+] Saved Golden Benchmark Report to: {report_csv_path}")


def plot_loss_curves(history):
    plt.figure(figsize=(10, 5))
    epochs_range = range(1, len(history["train_loss"]) + 1)
    
    plt.plot(epochs_range, history["train_loss"], label="Train Loss (LLM Dataset)", color="#1f77b4", linewidth=2)
    plt.plot(epochs_range, history["val_loss"], label="Val Loss (LLM Dataset)", color="#ff7f0e", linewidth=2)
    
    plt.title("Enhanced DINOv2 Fine-Tuning: Training vs. Validation Loss", fontsize=14, fontweight="bold")
    plt.xlabel("Total Iterated Epochs Across Folds", fontsize=12)
    plt.ylabel("BCE Loss", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(fontsize=12)
    plt.tight_layout()
    
    graph_path = OUTPUT_DIR / "loss_curves.png"
    plt.savefig(graph_path, dpi=300)
    plt.show()
    print(f"[+] Saved Loss Graph plot to: {graph_path}")


In [18]:
run_pipeline()

Initializing Enhanced DINOv2 Pipeline on cuda
[+] Loaded 58 Radiologist Golden Benchmark studies from COMPETITION_ROOT
[+] Loaded 4349 LLM-labeled training studies (after removing golden overlaps)

==================== FOLD 1/3 ====================
Ep 01/10 | Train Loss: 0.5675 | Val Loss: 0.5619 | Val AUC: 0.7515 | Time: 1339.5s
Ep 02/10 | Train Loss: 0.5415 | Val Loss: 0.5297 | Val AUC: 0.7832 | Time: 1365.5s
Ep 03/10 | Train Loss: 0.5219 | Val Loss: 0.5174 | Val AUC: 0.8055 | Time: 1330.0s
Ep 04/10 | Train Loss: 0.5087 | Val Loss: 0.5182 | Val AUC: 0.8057 | Time: 1350.8s
Ep 05/10 | Train Loss: 0.4968 | Val Loss: 0.5102 | Val AUC: 0.8147 | Time: 1415.4s
Ep 06/10 | Train Loss: 0.4874 | Val Loss: 0.5128 | Val AUC: 0.8162 | Time: 1430.2s


KeyboardInterrupt: 

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

def analyze_series_slice_counts(series_csv_path, series_dir_path=None):
    """
    Computes summary statistics (mean, median, max, min, std) for slice counts 
    per MRI series and study.
    """
    series_df = pd.read_csv(series_csv_path)
    
    # 1. Use pre-existing column or scan directory if missing
    if "n_slices" in series_df.columns:
        counts = series_df["n_slices"].values
    elif series_dir_path is not None:
        print("[*] 'n_slices' column missing. Scanning DICOM files on disk...")
        series_dir = Path(series_dir_path)
        counts = []
        for _, row in series_df.iterrows():
            s_path = series_dir / str(row["StudyInstanceUID"]) / str(row["SeriesInstanceUID"])
            if s_path.is_dir():
                counts.append(len(list(s_path.glob("*.dcm"))))
        counts = np.array(counts)
    else:
        raise ValueError("Either 'n_slices' column must exist in series_csv or series_dir_path must be provided.")

    # 2. Overall Per-Series Statistics
    print("=" * 60)
    print("           PER-SERIES SLICE COUNT STATISTICS           ")
    print("=" * 60)
    print(f"Total Series Evaluated : {len(counts):,}")
    print(f"Mean Slices per Series : {np.mean(counts):.2f}")
    print(f"Median Slices          : {np.median(counts):.1f}")
    print(f"Max Slices             : {np.max(counts)}")
    print(f"Min Slices             : {np.min(counts)}")
    print(f"Standard Deviation     : {np.std(counts):.2f}")
    print("-" * 60)

    # 3. Breakdown Per Anatomical Plane (if available)
    if "Anatomical_Plane" in series_df.columns:
        series_df["_computed_slices"] = counts
        plane_stats = series_df.groupby("Anatomical_Plane")["_computed_slices"].agg(
            ["count", "mean", "median", "max", "min"]
        )
        print("\nBREAKDOWN BY ANATOMICAL PLANE:")
        print(plane_stats.to_string())
        print("=" * 60)

    # 4. Total Slices per Study (Volume Stack)
    if "StudyInstanceUID" in series_df.columns:
        series_df["_computed_slices"] = counts
        study_totals = series_df.groupby("StudyInstanceUID")["_computed_slices"].sum().values
        print("\nTOTAL SLICES PER STUDY (FULL VOLUME STACK):")
        print(f"Mean Slices per Study  : {np.mean(study_totals):.2f}")
        print(f"Median Slices          : {np.median(study_totals):.1f}")
        print(f"Max Total Slices       : {np.max(study_totals)}")
        print(f"Min Total Slices       : {np.min(study_totals)}")
        print("=" * 60)

# Example Usage:
analyze_series_slice_counts(
    series_csv_path="/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv",
    series_dir_path="/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"
)

[*] 'n_slices' column missing. Scanning DICOM files on disk...
           PER-SERIES SLICE COUNT STATISTICS           
Total Series Evaluated : 24,371
Mean Slices per Series : 33.61
Median Slices          : 30.0
Max Slices             : 320
Min Slices             : 11
Standard Deviation     : 26.57
------------------------------------------------------------

BREAKDOWN BY ANATOMICAL PLANE:
                  count       mean  median  max  min
Anatomical_Plane                                    
Axial              5898  39.820448    32.0  254   12
Coronal            8609  28.269718    30.0   80   11
Sagittal           9864  34.554238    30.0  320   11

TOTAL SLICES PER STUDY (FULL VOLUME STACK):
Mean Slices per Study  : 185.86
Median Slices          : 162.0
Max Total Slices       : 632
Min Total Slices       : 67
